In [1]:
!pip install --no-index pandas

Looking in links: /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2023/x86-64-v4, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2023/x86-64-v3, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2023/generic, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/generic


In [2]:
import pandas as pd
import re
from pathlib import Path

In [3]:
# Finetune Models
# sh ./scripts/llama_7B_Dora.sh 4 8 ./finetuned_result/llama_7B/dora_r4 0
# sh ./scripts/llama_7B_Dora.sh 8 16 ./finetuned_result/llama_7B/dora_r8 0
# sh ./scripts/llama_7B_Dora.sh 16 32 ./finetuned_result/llama_7B/dora_r16 0
# sh ./scripts/llama_7B_Dora.sh 32 64 ./finetuned_result/llama_7B/dora_r32 0
# sh ./scripts/llama_7B_Dora.sh 64 128 ./finetuned_result/llama_7B/dora_r64 0

# sh ./scripts/llama_7B_Dora_qkv.sh 32 64 ./finetuned_result/llama_7B_dora_qkv_r32 0

# sh ./scripts/llama2_7B_Dora.sh 16 32 ./finetuned_result/llama2_7B/dora_r16 0
# sh ./scripts/llama2_7B_Dora.sh 32 64 ./finetuned_result/llama2_7B/dora_r32 0

# sh ./scripts/llama3_8B_Dora.sh 16 32 ./finetuned_result/llama3_8B/dora_r16 0
# sh ./scripts/llama3_8B_Dora.sh 32 64 ./finetuned_result/llama3_8B/dora_r32 0

In [4]:
# Evaluate models
# sh ./scripts/llama_7B_Dora_eval.sh ./finetuned_result/llama_7B/dora_r4 0
# sh ./scripts/llama_7B_Dora_eval.sh ./finetuned_result/llama_7B/dora_r8 0
# sh ./scripts/llama_7B_Dora_eval.sh ./finetuned_result/llama_7B/dora_r16 0
# sh ./scripts/llama_7B_Dora_eval.sh ./finetuned_result/llama_7B/dora_r32 0
# sh ./scripts/llama_7B_Dora_eval.sh ./finetuned_result/llama_7B/dora_r64 0

# sh ./scripts/llama_7B_Dora_eval.sh ./finetuned_result/llama_7B/llama_7B_dora_qkv_r32 0

# sh ./scripts/llama2_7B_Dora_eval.sh ./finetuned_result/llama2_7B/dora_r16 0
# sh ./scripts/llama2_7B_Dora_eval.sh ./finetuned_result/llama2_7B/dora_r32 0

# sh ./scripts/llama3_8B_Dora_eval.sh ./finetuned_result/llama3_8B/dora_r16 0
# sh ./scripts/llama3_8B_Dora_eval.sh ./finetuned_result/llama3_8B/dora_r32 0

In [3]:
def get_accuracy(text_file_path):
    pattern = re.compile(r"accuracy\s+\d+\s+([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)")

    with open(text_file_path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    if not lines:
        raise ValueError("Empty file")

    # check if the last line contains "finish"
    if "finish" not in lines[-1].strip().lower():
        raise ValueError("No finish marker in last line")

    for line in reversed(lines):
        m = pattern.search(line)
        if m:
            return float(m.group(1))

    raise ValueError("Non accuracy find")

In [4]:
base_dir = Path("/home/hwan783/scratch/homework/DoRA/commonsense_reasoning/finetuned_result/")
model_dict = {
    "llama_7B": ["dora_r4", "dora_r8", "dora_r16", "dora_r32", "dora_r64"],
    "llama2_7B": ["dora_r16", "dora_r32"],
    "llama3_8B": ["dora_r16", "dora_r32"]
}
dataset_kinds = ["ARC-Challenge", "ARC-Easy", "boolq", "hellaswag", "openbookqa", "piqa", "social_i_qa", "winogrande"]

In [5]:
data_paths = [
    base_dir / key / value 
    for key, item in model_dict.items()
    for value in item
]
data_paths

[PosixPath('/home/hwan783/scratch/homework/DoRA/commonsense_reasoning/finetuned_result/llama_7B/dora_r4'),
 PosixPath('/home/hwan783/scratch/homework/DoRA/commonsense_reasoning/finetuned_result/llama_7B/dora_r8'),
 PosixPath('/home/hwan783/scratch/homework/DoRA/commonsense_reasoning/finetuned_result/llama_7B/dora_r16'),
 PosixPath('/home/hwan783/scratch/homework/DoRA/commonsense_reasoning/finetuned_result/llama_7B/dora_r32'),
 PosixPath('/home/hwan783/scratch/homework/DoRA/commonsense_reasoning/finetuned_result/llama_7B/dora_r64'),
 PosixPath('/home/hwan783/scratch/homework/DoRA/commonsense_reasoning/finetuned_result/llama2_7B/dora_r16'),
 PosixPath('/home/hwan783/scratch/homework/DoRA/commonsense_reasoning/finetuned_result/llama2_7B/dora_r32'),
 PosixPath('/home/hwan783/scratch/homework/DoRA/commonsense_reasoning/finetuned_result/llama3_8B/dora_r16'),
 PosixPath('/home/hwan783/scratch/homework/DoRA/commonsense_reasoning/finetuned_result/llama3_8B/dora_r32')]

In [6]:
df = pd.DataFrame()
# Remove index name and "dataset" label
df.index.name = None
df.columns.name = None

# Drop model_rank as a column if it crept in, reset index to drop it
df = df.reset_index(drop=True)  # drop the model_rank index

# Re-build with cleaner index using just the row order
rows_list = []
for model, ranks in model_dict.items():
    for rank in ranks:
        rank_label = next(part for part in rank.split("_") if part.startswith("r"))
        row = {"base_model": model, "rank": int(rank_label[1:])}  # r32 -> 32

        for dataset in dataset_kinds:
            txt_path = base_dir / model / rank / f"{dataset}.txt"
            try:
                row[dataset] = get_accuracy(txt_path)
            except (ValueError, FileNotFoundError) as e:
                row[dataset] = None
                print(f"  Warning: {e} for {txt_path}")

        rows_list.append(row)

df = pd.DataFrame(rows_list)

# Average column: treat None as 0
df["average"] = df[dataset_kinds].apply(
    lambda r: r.fillna(0).mean(), axis=1
)

df

,base_model,rank,ARC-Challenge,ARC-Easy,boolq,hellaswag,openbookqa,piqa,social_i_qa,winogrande,average
0,llama_7B,4,0.626280,0.786195,0.401223,0.251544,0.786,0.420022,0.778403,0.787687,0.604669
1,llama_7B,8,0.656997,0.816077,0.698165,0.851723,0.798,0.818825,0.797339,0.801894,0.779877
2,llama_7B,16,0.654437,0.806397,0.699388,0.831707,0.776,0.824810,0.795803,0.806630,0.774397
3,llama_7B,32,0.453925,0.505892,0.681957,0.667297,0.578,0.809576,0.477482,0.439621,0.576719
4,llama_7B,64,0.593003,0.744949,0.700306,0.784206,0.742,0.819369,0.693961,0.786898,0.733087
5,llama2_7B,16,0.710751,0.843434,0.720183,0.890360,0.812,0.831338,0.799386,0.829519,0.804622
6,llama2_7B,32,0.684300,0.836279,0.717431,0.890858,0.824,0.837867,0.759980,0.825572,0.797036
7,llama3_8B,16,0.789249,0.901094,0.744954,0.954989,0.872,0.887378,0.803480,0.846882,0.850003
8,llama3_8B,32,0.803754,0.905303,0.746177,0.955089,0.860,0.892818,0.799386,0.855564,0.852261
